In [1]:
from gplearn.genetic import SymbolicTransformer
from gplearn.genetic import SymbolicRegressor
from gplearn.functions import make_function
from sklearn.utils.random import check_random_state

import numpy as np
import array
import pandas
import matplotlib
import math   # 导入 math 模块
import matplotlib.pyplot as plt
import pandas as pd
from pandas import Series, DataFrame
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from itertools import combinations
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
OL= LinearRegression(normalize=False)

plt.rcParams['savefig.dpi'] = 100 #图片像素
plt.rcParams['figure.dpi'] = 100#分辨率

data=pandas.read_csv(r"C:\Users\z\Desktop\模型统计性质分析\二元合金体积效应数据分类分析\Outliers\Select by Pearson cofficients from initial dataset.csv")

In [2]:
F=np.array(data)
p=data.ev_ratio
p=np.array(p)
F_0=F[:,3:67]

In [3]:
F_0.shape

(418, 64)

In [5]:
crossover=[]
parsimony=[]
for i in range(25,85,2):
    crossover.append(i*0.01)

for i in range(5,55,10):
    parsimony.append(i*0.00002)
    
crossover_2=[]
subtree_mutation=[]
hoist_mutation=[]

for k in range(len(crossover)):
    up=math.floor((1-crossover[k])*100/3)
    low=math.floor((0.92-crossover[k])*100/3)
    for i in range(low,up,15):
        subtree_mutation.append(i*0.01)
        hoist_mutation.append(i*0.01)
        crossover_2.append(crossover[k])
        
point_mutation=[]
point_mutation=np.array(1-np.array(crossover_2)-np.array(subtree_mutation)-np.array(hoist_mutation))
tot=np.c_[crossover_2,subtree_mutation,hoist_mutation,point_mutation]

In [15]:
ss = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)

tr=[]
te=[]

for train_index, test_index in ss.split(F_0):
    tr.append(train_index)
    te.append(test_index)

In [20]:
function_set = [ 'add','sub','mul', 'div', 'sqrt', 'log','abs','neg', 'inv','sin','cos','tan']

n=4

F_tr=F_0[tr[n],:]
F_te=F_0[te[n],:]

p_tr=p[tr[n]]
p_te=p[te[n]]

j=len(parsimony)

gen=44#len(tot)

for m in range(0,1):
    for i in range(17,18):
        gp = SymbolicTransformer(population_size=10500,tournament_size=20,
                                 metric='pearson',function_set = function_set,init_depth=(4, 15),
                                 generations=gen, stopping_criteria=0.9,
                                 hall_of_fame=500, n_components=1,
                                 p_crossover=tot[i][0],p_subtree_mutation=tot[i][1],
                                 p_hoist_mutation=tot[i][2],p_point_mutation=tot[i][3],
                                 max_samples=0.9, verbose=1,n_jobs=3,
                                 parsimony_coefficient=parsimony[m], random_state=0)

        gp.fit(F_tr,(p_tr.reshape(-1,1)-1)*100)
        
        for program in gp:
            print(program)
            print(program.length_)
            print(program.raw_fitness_)
        
        gp_features = gp.transform(F_tr)
        OL.fit(gp_features, (p_tr.reshape(-1,1)-1)*100)
    
        gp_features_te = gp.transform(F_te)
        p_pre=OL.predict(gp_features_te)
        r_2=r2_score((p_te.reshape(-1,1)-1)*100, p_pre)
        result=pd.DataFrame(np.c_[(p_te.reshape(-1,1)-1)*100,p_pre])
        pearson=result.corr('pearson')
        
        print(pearson[0][1])
        print(r_2)

D:\Python\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    49.76        0.0840665        2         0.575371         0.241967      6.71m
   1     7.78         0.261564        4         0.704208         0.474056      4.06m
   2     3.70         0.422618        5         0.706865         0.499146      4.08m
   3     3.09         0.479343        4         0.704597         0.652285      3.93m
   4     4.58         0.477613        5         0.725796         0.391181      3.99m
   5     4.82         0.507345        5         0.732736         0.566009      4.01m
   6     5.03          0.51437        6          0.74178         0.366406      3.81m
   7     5.47          0.51785        5         0.737264         0.321508      3.74m
   8     5.76         0.514323        6         0.741653          0.31546  

D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
